# Capacity Watchdog — baseline dinâmica 7d + kill escalonado

**O que faz:** a cada execução (agendar a cada 10–15 min), tira um snapshot do consumo de CU por item
via semantic model do **Fabric Capacity Metrics App**, calcula o consumo do intervalo (diff entre snapshots),
compara contra a **média dos últimos 7 dias no mesmo bucket horário** e age em degraus:

| Degrau | Condição (consumo ÷ baseline) | Ação |
|---|---|---|
| T1 | > 1.2× | Alerta (Teams) |
| T2 | > 1.5× | Cancela refreshes e jobs em background do item |
| T3 | > 1.8× | Também mata sessões XMLA interativas do dataset |

**Pré-requisitos:**
1. Capacity Metrics App instalado; anote o nome do workspace e do dataset dele.
2. Identidade que executa o notebook precisa ser **capacity admin** + admin dos workspaces monitorados.
3. Rodar primeiro em `MODE = "OBSERVE"` por 2–4 semanas antes de armar `ENFORCE`.


In [ ]:
# ============ CELL 1 — CONFIG ============
MODE = "OBSERVE"          # "OBSERVE" = só loga/alerta | "ENFORCE" = executa kills
CAPACITIES = {
    # nome amigável : capacity_id (GUID — pegue no Admin Portal ou via API)
    "F128_PROD":    "<capacity-guid-f128>",
    "F64_SANDBOX":  "<capacity-guid-f64>",
}

# Workspace/dataset do Capacity Metrics App (ajuste se renomeou)
METRICS_WORKSPACE = "Microsoft Fabric Capacity Metrics"
METRICS_DATASET   = "Fabric Capacity Metrics"

# Degraus (razão consumo_intervalo / baseline_7d)
TIER_ALERT        = 1.2
TIER_KILL_BG      = 1.5
TIER_KILL_SESSION = 1.8

# Itens que NUNCA sofrem kill (regulatório, executivo). Alerta continua valendo.
ALLOWLIST_ITEM_IDS = {
    # "<item-guid-relatorio-antt>",
}

# Ignora anomalia se consumo absoluto do intervalo for irrisório (evita falso positivo
# em itens novos/ociosos: 3 CUs vs baseline 1 CU = razão 3.0, mas irrelevante)
MIN_CU_SECONDS = 300

# Mínimo de dias com histórico no bucket para a baseline valer (senão só observa)
MIN_BASELINE_DAYS = 4

# Persistência (Lakehouse anexado ao notebook)
SNAPSHOT_TABLE = "watchdog_snapshots"
EVENTS_TABLE   = "watchdog_events"

TEAMS_WEBHOOK_URL = ""    # opcional: incoming webhook p/ alertas

INTERVAL_MIN = 15         # deve bater com a frequência do agendamento


In [ ]:
# ============ CELL 2 — AUTH / HELPERS ============
import requests, json, datetime as dt
import sempy.fabric as fabric
from pyspark.sql import functions as F, Window

def token_pbi():
    return notebookutils.credentials.getToken("https://analysis.windows.net/powerbi/api")

def token_fabric():
    return notebookutils.credentials.getToken("https://api.fabric.microsoft.com")

def _hdr(tok):
    return {"Authorization": f"Bearer {tok}", "Content-Type": "application/json"}

NOW = dt.datetime.utcnow()
HOUR_BUCKET = NOW.hour          # baseline por hora do dia
DOW = NOW.weekday()             # opcional: refinar por dia da semana

def log_event(level, capacity, item_id, item_name, ratio, cu, baseline, action, detail=""):
    row = [(NOW, level, capacity, item_id, item_name,
            float(ratio or 0), float(cu or 0), float(baseline or 0), action, detail, MODE)]
    cols = ["ts","level","capacity","item_id","item_name","ratio","cu_s","baseline_cu_s","action","detail","mode"]
    spark.createDataFrame(row, cols).write.mode("append").saveAsTable(EVENTS_TABLE)
    print(f"[{level}] {capacity} | {item_name} | ratio={ratio:.2f} cu={cu:.0f} base={baseline:.0f} -> {action} {detail}")


### Cell 3 — Exploração (rode uma vez, depois pode pular)
Os nomes de tabelas/medidas do Capacity Metrics App **variam por versão**. Use esta célula
para descobrir os nomes reais e ajustar o DAX da Cell 4.

In [ ]:
# ============ CELL 3 — EXPLORAR MODELO DO METRICS APP (one-off) ============
# display(fabric.list_tables(dataset=METRICS_DATASET, workspace=METRICS_WORKSPACE))
# display(fabric.list_measures(dataset=METRICS_DATASET, workspace=METRICS_WORKSPACE))
# Procure a tabela de métricas por item/dia (ex.: 'MetricsByItemandOperationandDay')
# e a dimensão de itens (ex.: 'Items') e capacidades ('Capacities').


In [ ]:
# ============ CELL 4 — SNAPSHOT: CU acumulado HOJE por item ============
# Estratégia: em vez de depender de granularidade horária do app (instável entre versões),
# lemos o ACUMULADO DO DIA por item a cada execução e derivamos o consumo do intervalo
# via diff contra o snapshot anterior (Cell 5).

DAX_TEMPLATE = """
EVALUATE
SUMMARIZECOLUMNS(
    Items[ItemId],
    Items[ItemName],
    Items[ItemKind],
    Items[WorkspaceId],
    Items[WorkspaceName],
    FILTER(VALUES(Capacities[capacityId]), Capacities[capacityId] = "{cap_id}"),
    FILTER(VALUES(MetricsByItemandOperationandDay[Date]), MetricsByItemandOperationandDay[Date] = TODAY()),
    "cu_s_today", CALCULATE(SUM(MetricsByItemandOperationandDay[sum_CU]))
)
"""
# ^ AJUSTE nomes de tabela/coluna/medida conforme Cell 3.
#   Alternativa comum: medida [CU (s)] no lugar de SUM(sum_CU).

snap_frames = []
for cap_name, cap_id in CAPACITIES.items():
    df = fabric.evaluate_dax(
        dataset=METRICS_DATASET, workspace=METRICS_WORKSPACE,
        dax_string=DAX_TEMPLATE.format(cap_id=cap_id),
    )
    if df is None or len(df) == 0:
        continue
    df.columns = ["item_id","item_name","item_kind","workspace_id","workspace_name","cu_s_today"]
    df["capacity"] = cap_name
    snap_frames.append(df)

import pandas as pd
snap_pd = pd.concat(snap_frames, ignore_index=True) if snap_frames else pd.DataFrame()
snap_pd["ts"] = NOW
snap_pd["date"] = NOW.date()
snap_pd["hour_bucket"] = HOUR_BUCKET

snap = spark.createDataFrame(snap_pd)
snap.write.mode("append").saveAsTable(SNAPSHOT_TABLE)
print(f"Snapshot: {snap.count()} itens em {len(CAPACITIES)} capacidade(s)")


In [ ]:
# ============ CELL 5 — CONSUMO DO INTERVALO + BASELINE 7D ============
hist = spark.table(SNAPSHOT_TABLE)

# Consumo do intervalo = acumulado atual - acumulado do snapshot anterior (mesmo item, mesmo dia)
w = Window.partitionBy("capacity","item_id","date").orderBy("ts")
interval = (hist
    .withColumn("prev_cu", F.lag("cu_s_today").over(w))
    .withColumn("cu_interval", F.col("cu_s_today") - F.coalesce(F.col("prev_cu"), F.lit(0.0)))
    .withColumn("cu_interval", F.when(F.col("cu_interval") < 0, F.col("cu_s_today"))  # reset diário
                                .otherwise(F.col("cu_interval")))
)

# Baseline: média do cu_interval nos últimos 7 dias, mesmo item, mesmo hour_bucket (exclui hoje)
today = NOW.date()
lookback = today - dt.timedelta(days=7)
baseline = (interval
    .filter((F.col("date") >= F.lit(lookback)) & (F.col("date") < F.lit(today)))
    .filter(F.col("hour_bucket") == HOUR_BUCKET)
    .groupBy("capacity","item_id")
    .agg(F.avg("cu_interval").alias("baseline_cu_s"),
         F.countDistinct("date").alias("baseline_days"))
)

# Consumo atual = último intervalo de hoje por item
latest_ts = interval.filter(F.col("date") == F.lit(today)).agg(F.max("ts")).collect()[0][0]
current = (interval
    .filter((F.col("date") == F.lit(today)) & (F.col("ts") == F.lit(latest_ts)))
    .select("capacity","item_id","item_name","item_kind","workspace_id","workspace_name","cu_interval")
)

scored = (current.join(baseline, ["capacity","item_id"], "left")
    .withColumn("ratio", F.col("cu_interval") / F.col("baseline_cu_s"))
)
display(scored.orderBy(F.desc("ratio")))


In [ ]:
# ============ CELL 6 — DETECÇÃO ============
anomalies = (scored
    .filter(F.col("cu_interval") >= MIN_CU_SECONDS)
    .filter(F.col("baseline_days") >= MIN_BASELINE_DAYS)
    .filter(F.col("ratio") >= TIER_ALERT)
).collect()

print(f"{len(anomalies)} anomalia(s) acima de {TIER_ALERT}x")


In [ ]:
# ============ CELL 7 — AÇÕES DE KILL ============

def cancel_refreshes(workspace_id, dataset_id):
    """Cancela refreshes em andamento de um semantic model (Enhanced Refresh API)."""
    tok = token_pbi()
    base = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/refreshes"
    r = requests.get(base + "?$top=5", headers=_hdr(tok)); r.raise_for_status()
    killed = []
    for ref in r.json().get("value", []):
        if ref.get("status") == "Unknown":  # "Unknown" = em andamento na API de refresh
            rid = ref["requestId"]
            resp = requests.delete(f"{base}/{rid}", headers=_hdr(tok))
            killed.append((rid, resp.status_code))
    return killed

def cancel_fabric_jobs(workspace_id, item_id):
    """Cancela job instances em execução (notebooks, pipelines, dataflows gen2)."""
    tok = token_fabric()
    base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{item_id}/jobs/instances"
    r = requests.get(base, headers=_hdr(tok))
    if r.status_code != 200:
        return []
    killed = []
    for job in r.json().get("value", []):
        if job.get("status") in ("InProgress", "NotStarted"):
            jid = job["id"]
            resp = requests.post(f"{base}/{jid}/cancel", headers=_hdr(tok))
            killed.append((jid, resp.status_code))
    return killed

def kill_xmla_sessions(workspace_name, dataset_name):
    """Mata sessões XMLA ativas do dataset (queries interativas). USE COM CAUTELA."""
    tom = fabric.create_tom_server(readonly=False, workspace=workspace_name)
    sessions = fabric.evaluate_dax(
        dataset=dataset_name, workspace=workspace_name,
        dax_string="EVALUATE $SYSTEM.DISCOVER_SESSIONS")  # se falhar, use INFO/DMV via read_table
    killed = []
    for _, s in sessions.iterrows():
        spid = s.get("SESSION_SPID") or s.get("[SESSION_SPID]")
        if not spid:
            continue
        xmla = f'''<Cancel xmlns="http://schemas.microsoft.com/analysisservices/2003/engine">
                     <SPID>{int(spid)}</SPID><CancelAssociated>true</CancelAssociated></Cancel>'''
        try:
            tom.Execute(xmla)
            killed.append(int(spid))
        except Exception as e:
            print(f"  falha ao matar SPID {spid}: {e}")
    tom.Disconnect()
    return killed


In [ ]:
# ============ CELL 8 — ORQUESTRAÇÃO ============
def notify_teams(text):
    if not TEAMS_WEBHOOK_URL:
        return
    try:
        requests.post(TEAMS_WEBHOOK_URL, json={"text": text}, timeout=10)
    except Exception as e:
        print(f"Teams webhook falhou: {e}")

for a in anomalies:
    ratio, cu, base = a["ratio"], a["cu_interval"], a["baseline_cu_s"]
    protected = a["item_id"] in ALLOWLIST_ITEM_IDS
    tag = " [ALLOWLIST]" if protected else ""

    # T1 — sempre alerta
    log_event("T1_ALERT", a["capacity"], a["item_id"], a["item_name"], ratio, cu, base,
              "alert", tag)
    notify_teams(f"⚠️ [{a['capacity']}] {a['item_name']} a {ratio:.1f}x da baseline "
                 f"({cu:.0f} vs {base:.0f} CU·s / {INTERVAL_MIN}min){tag}")

    if protected or MODE != "ENFORCE":
        continue

    # T2 — mata background
    if ratio >= TIER_KILL_BG:
        killed = []
        if a["item_kind"] in ("Dataset", "SemanticModel"):
            killed += [f"refresh:{x}" for x in cancel_refreshes(a["workspace_id"], a["item_id"])]
        killed += [f"job:{x}" for x in cancel_fabric_jobs(a["workspace_id"], a["item_id"])]
        log_event("T2_KILL_BG", a["capacity"], a["item_id"], a["item_name"], ratio, cu, base,
                  "cancel_background", str(killed))
        notify_teams(f"🛑 [{a['capacity']}] Background cancelado: {a['item_name']} "
                     f"({ratio:.1f}x) — {killed}")

    # T3 — mata sessões interativas
    if ratio >= TIER_KILL_SESSION and a["item_kind"] in ("Dataset", "SemanticModel"):
        spids = kill_xmla_sessions(a["workspace_name"], a["item_name"])
        log_event("T3_KILL_SESSION", a["capacity"], a["item_id"], a["item_name"], ratio, cu, base,
                  "cancel_sessions", str(spids))
        notify_teams(f"🔴 [{a['capacity']}] Sessões mortas em {a['item_name']}: SPIDs {spids}")

print("Watchdog concluído.")


## Operação

**Agendamento:** agende este notebook a cada **15 min** (bata com `INTERVAL_MIN`). Rode-o
numa capacidade *diferente* da monitorada se possível (ou aceite que o próprio watchdog consome CU — é leve).

**Calibração (fase OBSERVE, 2–4 semanas):**
1. Consulte `watchdog_events` e conte falsos positivos por item/horário.
2. Ajuste `MIN_CU_SECONDS`, os tiers e popule `ALLOWLIST_ITEM_IDS` com os itens críticos
   (regulatório ANTT, executivos) — eles alertam mas nunca são mortos.
3. Se o padrão semanal for forte (fechamento de mês), refine a baseline filtrando também por
   `dayofweek` na Cell 5 — o esqueleto já grava `hour_bucket`; adicionar DOW é uma linha.

**Só então** mude `MODE = "ENFORCE"` — e comece armando apenas na F64. A F128 fica em OBSERVE
até você confiar no comportamento.

**Limitações conhecidas:**
- O Capacity Metrics App tem latência de alguns minutos — o kill nunca é instantâneo.
- `cancel_refreshes` só enxerga refreshes disparados via API/agendamento; refresh manual pelo
  portal aparece igualmente na lista, mas o status "Unknown" é o indicador de em-andamento.
- `kill_xmla_sessions` derruba usuários no meio do relatório. T3 deve ser raro por design.
- Nomes de tabelas/medidas do Metrics App variam por versão — valide na Cell 3 antes do primeiro run.
